In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import json
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from results_loader import METHODS, load_results

plt.rcParams.update({
    "text.usetex": True,
    "font.size": 20,
})

In [2]:
methods = list(METHODS)
records = load_results("../outputs/results", dataset="harvard", asr_model="whisper-tiny")  # aligned by sentence across methods
results = [dict() for _ in methods]

## Runtime & Budget

In [5]:
jsons = [[r["summary_path"] for r in records[m]] for m in methods]

In [6]:
# Runtime: end-to-end wall clock per sentence; budget: generations x population size
for m, r in zip(methods, results):
    times = [rec["elapsed_seconds"] for rec in records[m]]
    budget = [rec["budget"] for rec in records[m]]
    r |= {"runt": times, "budget": budget}
    print(f"{m} - Runtime: {np.mean(times):.2f} pm {np.std(times):.2f} s")
    print(f"{m} - Budget: {np.mean(budget):.2f} pm {np.std(budget):.2f}")

    ## Statistical Test

In [7]:
alternative = {
    "runt": "less",
    "budget": "less",
}

In [8]:
def cohens_d_paired(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    n = min(len(a), len(b))
    diff = a[:n] - b[:n]
    return abs(np.mean(diff) / np.std(diff, ddof=1))

In [9]:
for i, m in enumerate(results[1:], 1):
    print(f"{methods[0]} vs {methods[i]}")
    for k, vals in results[0].items():
        stat, p = wilcoxon(vals, m[k], alternative=alternative[k])
        d = cohens_d_paired(vals, m[k])
        print(f"\t{k}: p={p:.3e}; Cohens-D: {d:.1f}")

## Convergence Plot

In [12]:
fig, ax = plt.subplots(figsize=(8, 5))
gens = np.arange(1, 101)

# GATAS and Waveform report the generation at which they stopped (thresholds met or budget used)
for method in ["GATAS", "Waveform"]:
    curves = []
    for r in records[method]:
        if r["success"]:
            cumul = (gens >= int(r["generations"])).astype(float)
        else:
            cumul = np.zeros_like(gens, dtype=float)
        curves.append(cumul)

    X = np.vstack(curves)
    mean = np.nanmean(X, axis=0)

    ax.plot(gens, mean, label=method)
    ax.fill_between(gens, 0, mean, alpha=0.2)

ax.set_xlabel("Generation")
ax.set_ylabel("Cumulative Success Rate")
ax.set_xlim(1, 100)
ax.set_ylim(0, 1)
ax.legend()
ax.legend(loc="upper left")
ax.grid(axis="y")
ax.set_xlim([1,100])

plt.savefig("_figures/convergance.pdf", dpi=200, bbox_inches="tight")